In [12]:
import pandas as pd
import matplotlib.pyplot as plt 
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import seaborn as sns
from pathlib import Path
import joblib


In [13]:
pbp_DF = pd.read_csv('../datasets/props_training.csv')
pbp_DF.head()

,game_id,player_id,season,seconds_remaining,current_points,current_minutes,season_ppg,season_fga,season_mpg,fga_so_far,3pa_so_far,season_3pa,current_rebounds,season_rebounds,current_assists,season_assists,score_differential,final_points,final_rebounds,final_assists
0,22100001,200746,2021-22,2255,0,3.63,12.91,9.74,22.34,0,0,0.98,1,5.49,0,0.89,15,1,3,1
1,22100001,200746,2021-22,1647,0,8.88,12.91,9.74,22.34,0,0,0.98,2,5.49,0,0.89,16,1,3,1
2,22100001,200746,2021-22,1617,0,9.38,12.91,9.74,22.34,0,0,0.98,3,5.49,0,0.89,13,1,3,1
3,22100001,200746,2021-22,1121,0,11.73,12.91,9.74,22.34,0,0,0.98,3,5.49,1,0.89,8,1,3,1
4,22100001,200746,2021-22,1018,1,13.45,12.91,9.74,22.34,0,0,0.98,3,5.49,1,0.89,11,1,3,1


In [14]:
pbp_DF['points_remaining'] = pbp_DF['final_points'] - pbp_DF['current_points']
pbp_DF.drop(columns=['game_id', 'player_id', 'season', 'current_rebounds', 'season_rebounds', 'current_assists', 'season_assists', 'final_rebounds', 'final_assists', 'final_points'], inplace=True)
pbp_DF.drop_duplicates(inplace=True)
pbp_DF.head()

,seconds_remaining,current_points,current_minutes,season_ppg,season_fga,season_mpg,fga_so_far,3pa_so_far,season_3pa,score_differential,points_remaining
0,2255,0,3.63,12.91,9.74,22.34,0,0,0.98,15,1
1,1647,0,8.88,12.91,9.74,22.34,0,0,0.98,16,1
2,1617,0,9.38,12.91,9.74,22.34,0,0,0.98,13,1
3,1121,0,11.73,12.91,9.74,22.34,0,0,0.98,8,1
4,1018,1,13.45,12.91,9.74,22.34,0,0,0.98,11,0


In [15]:
feature_cols = ['seconds_remaining', 'current_points', 'current_minutes', 'season_ppg', 'season_fga', 'season_mpg', 'fga_so_far', '3pa_so_far', 'season_3pa', 'score_differential']
X = pbp_DF[feature_cols]
y = pbp_DF['points_remaining']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

In [16]:
model = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=500,
    max_depth=7,
    learning_rate=0.1,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42
)
model.fit(
    X_tr, y_tr,
    eval_set=[(X_val, y_val)],
    verbose=True
)

[0]	validation_0-rmse:7.57905
[1]	validation_0-rmse:7.17058
[2]	validation_0-rmse:6.82455
[3]	validation_0-rmse:6.54065
[4]	validation_0-rmse:6.28375
[5]	validation_0-rmse:6.06788
[6]	validation_0-rmse:5.88571
[7]	validation_0-rmse:5.73334
[8]	validation_0-rmse:5.60629
[9]	validation_0-rmse:5.50066
[10]	validation_0-rmse:5.41216
[11]	validation_0-rmse:5.33934
[12]	validation_0-rmse:5.27903
[13]	validation_0-rmse:5.22949
[14]	validation_0-rmse:5.18892
[15]	validation_0-rmse:5.15510
[16]	validation_0-rmse:5.12722
[17]	validation_0-rmse:5.10383
[18]	validation_0-rmse:5.08446
[19]	validation_0-rmse:5.06813
[20]	validation_0-rmse:5.05479
[21]	validation_0-rmse:5.04335
[22]	validation_0-rmse:5.03390
[23]	validation_0-rmse:5.02631
[24]	validation_0-rmse:5.01918
[25]	validation_0-rmse:5.01372
[26]	validation_0-rmse:5.00890
[27]	validation_0-rmse:5.00488
[28]	validation_0-rmse:5.00119
[29]	validation_0-rmse:4.99798
[30]	validation_0-rmse:4.99489
[31]	validation_0-rmse:4.99240
[32]	validation_0-

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,0.9
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes 

In [17]:
def eval_model():
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    print(f"MAE: {mae:.2f}")
    print(f"RMSE: {rmse:.2f}")
    print(f"R²: {r2:.3f}")

def custom_predictions(custom_data):
    # index=[0] required when custom_data is a dict of scalars (one row)
    custom_df = pd.DataFrame(custom_data, index=[0])
    custom_df = custom_df[['seconds_remaining', 'current_points', 'current_minutes', 'season_ppg', 'season_fga', 'season_mpg', 'fga_so_far', '3pa_so_far', 'season_3pa', 'score_differential']]
    points_remaining = round(model.predict(custom_df)[0])
    points_started = custom_data['current_points']
    print(f"Points started with: {points_started}")
    print(f"Points predicted left to score: {points_remaining}")
    print(f"Final points: {points_started + points_remaining}")

def save_model(model, model_path_name):
    model_path = Path(model_path_name)
    joblib.dump(model, model_path)
    print(f"Model saved to {model_path.absolute()}")
    

In [18]:
eval_model()

MAE: 3.56
RMSE: 23.48
R²: 0.636


In [19]:
custom_data = {
    'seconds_remaining': 2880,
    'current_points': 0,
    'current_minutes': 0,
    'season_ppg': 4,
    'season_fga': 2,
    'season_mpg': 17,
    'fga_so_far': 0,
    '3pa_so_far': 0,
    'season_3pa': 1,
    'score_differential': 0,
}

custom_predictions(custom_data)

Points started with: 0
Points predicted left to score: 4
Final points: 4


In [20]:
# More test cases
test_cases = [
    # Low scorer at game start (role player)
    {"name": "Low scorer, game start", "data": {
        "seconds_remaining": 2880, "current_points": 0, "current_minutes": 0,
        "season_ppg": 4, "season_fga": 2, "season_mpg": 17,
        "fga_so_far": 0, "3pa_so_far": 0, "season_3pa": 1, "score_differential": 0,
    }},
    # Star scorer at game start
    {"name": "Star scorer (28 ppg), game start", "data": {
        "seconds_remaining": 2880, "current_points": 0, "current_minutes": 0,
        "season_ppg": 28, "season_fga": 20, "season_mpg": 35,
        "fga_so_far": 0, "3pa_so_far": 0, "season_3pa": 8, "score_differential": 0,
    }},
    # Mid-game, player already has 15 pts (on pace)
    {"name": "Mid-game, 15 pts so far, ~20 ppg player", "data": {
        "seconds_remaining": 1440, "current_points": 15, "current_minutes": 18,
        "season_ppg": 20, "season_fga": 15, "season_mpg": 32,
        "fga_so_far": 10, "3pa_so_far": 4, "season_3pa": 6, "score_differential": 5,
    }},
    # Late game, 5 min left, star has 22
    {"name": "Late game, 5 min left, star with 22 pts", "data": {
        "seconds_remaining": 300, "current_points": 22, "current_minutes": 32,
        "season_ppg": 27, "season_fga": 19, "season_mpg": 35,
        "fga_so_far": 16, "3pa_so_far": 6, "season_3pa": 7, "score_differential": -3,
    }},
    # Game almost over, role player with 6 pts
    {"name": "60 sec left, role player with 6 pts", "data": {
        "seconds_remaining": 60, "current_points": 6, "current_minutes": 22,
        "season_ppg": 5, "season_fga": 3, "season_mpg": 18,
        "fga_so_far": 4, "3pa_so_far": 1, "season_3pa": 1.5, "score_differential": 15,
    }},
    # Blowout - bench player in garbage time
    {"name": "Blowout, bench player 2 pts, 8 min left", "data": {
        "seconds_remaining": 480, "current_points": 2, "current_minutes": 6,
        "season_ppg": 4, "season_fga": 2, "season_mpg": 12,
        "fga_so_far": 2, "3pa_so_far": 0, "season_3pa": 0.5, "score_differential": 25,
    }},
]

for tc in test_cases:
    print(f"\n--- {tc['name']} ---")
    custom_predictions(tc["data"])


--- Low scorer, game start ---
Points started with: 0
Points predicted left to score: 4
Final points: 4

--- Star scorer (28 ppg), game start ---
Points started with: 0
Points predicted left to score: 24
Final points: 24

--- Mid-game, 15 pts so far, ~20 ppg player ---
Points started with: 15
Points predicted left to score: 10
Final points: 25

--- Late game, 5 min left, star with 22 pts ---
Points started with: 22
Points predicted left to score: 5
Final points: 27

--- 60 sec left, role player with 6 pts ---
Points started with: 6
Points predicted left to score: 1
Final points: 7

--- Blowout, bench player 2 pts, 8 min left ---
Points started with: 2
Points predicted left to score: 3
Final points: 5


In [21]:
save_model(model, 'pts_model.joblib')

Model saved to /Users/lemons/Documents/universidad/cs/pj09-sports-betting/ml/models_PTS/pts_model.joblib
